In [36]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [37]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'meta' / 'runs.csv').exists():
            return candidate
    raise FileNotFoundError(f"Could not find meta/runs.csv above {start}")

repo_root = find_repo_root(Path.cwd())

runs = pd.read_csv(repo_root / 'meta' / 'runs.csv')
runs['valid'] = runs['valid'].astype(str).str.lower() == 'true'
runs['delidded'] = runs['delidded'].astype(str).str.lower() == 'true'
valid_runs = runs[runs['valid']]

print(f"meta/runs.csv lists {len(runs)} runs, {len(valid_runs)} marked valid")

meta/runs.csv lists 69 runs, 58 marked valid


In [38]:
# Plain Ppkg/Tdie-vs-sample plot for one run (downsampled to every 10th point)
# Tdie and Ppkg use separate y-axes, fixed to the same scale across all plots
def plot_run(file_path):
    # df = pd.read_csv(file_path).iloc[::10]
    df = pd.read_csv(file_path) 

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    if 'Tdie' in df.columns:
        fig.add_trace(go.Scatter(y=df['Tdie'], name='Tdie'), secondary_y=False)
    if 'Ppkg' in df.columns:
        fig.add_trace(go.Scatter(y=df['Ppkg'], name='Ppkg'), secondary_y=True)

    fig.update_layout(title=file_path.name, height=700)
    fig.update_xaxes(title_text='Sample')
    fig.update_yaxes(title_text='Tdie (°C)', range=[20, 100], secondary_y=False)
    fig.update_yaxes(title_text='Ppkg (W)', range=[50, 220], secondary_y=True)
    fig.show()

def plot_runs(runs_df):
    for _, run in runs_df.iterrows():
        plot_run(repo_root / run['path'].replace('\\', '/'))

## Baseline: stock CPU + ASUS ROG Ryuo 120

Stock (non-delidded) Ryzen 9 7900X paired with the ASUS ROG Ryuo 120 AIO — the AIO that fits in the target case without any modifications. This is the out-of-the-box baseline everything else is compared against.

In [39]:
baseline_runs = valid_runs[
    (~valid_runs['delidded']) & (valid_runs['loop_under_test'] == 'ASUS ROG Ryuo 120')
]
plot_runs(baseline_runs)

In [40]:
import numpy as np

# The real acquisition harness ramps Ppkg (PPT limit) in ~10W steps, holding each step
# until is_stable() (std < 0.3 degC, |slope| < 0.005 degC/s over the trailing window)
# returns True, then advances. The run only ends *without* such a transition when
# sustained thermal throttling aborts the whole test via KeyboardInterrupt -- so the
# LAST plateau in every file was never confirmed stable and is dropped here rather
# than treated as a genuine settled point.
def find_plateaus(df, step=10, min_len=120):
    p_round = (df['Ppkg'] / step).round() * step
    group_id = (p_round != p_round.shift()).cumsum()
    groups = [g for _, g in df.groupby(group_id) if len(g) >= min_len]
    return groups[:-1]

def settled_point(group, tail_n=60):
    tail = group.tail(tail_n)
    tdie_tail = tail['Tdie'].to_numpy()
    std = np.std(tdie_tail)
    slope = np.polyfit(np.arange(len(tdie_tail)), tdie_tail, 1)[0]
    settled_ppkg = tail['Ppkg'].mean()
    settled_tdie = np.median(tdie_tail) - tail['Tair_in'].mean() + 20
    return settled_ppkg, settled_tdie, std, slope

def settled_points_for_run(file_path, max_std=0.3, max_slope=0.005):
    df = pd.read_csv(file_path)
    rows = []
    for group in find_plateaus(df):
        power, tdie, std, slope = settled_point(group)
        if std > max_std or abs(slope) > max_slope:
            level = round(power / 10) * 10
            print(f"warning: unstable tail in {file_path.name} at ~{level:.0f}W (std={std:.3f}, slope={slope:+.5f})")
        rows.append((power, tdie))
    return pd.DataFrame(rows, columns=['power', 'tdie'])

def plot_settled(runs_df):
    fig = go.Figure()
    for _, run in runs_df.iterrows():
        file_path = repo_root / run['path'].replace('\\', '/')
        points = settled_points_for_run(file_path)
        fig.add_trace(go.Scatter(x=points['power'], y=points['tdie'], mode='lines+markers', name=file_path.name))

    fig.update_layout(title='Settled Tdie vs Ppkg', height=700)
    fig.update_xaxes(title_text='Ppkg (W)', range=[50, 220])
    fig.update_yaxes(title_text='Tdie (°C, ambient-normalized)', range=[20, 100])
    fig.show()

### Baseline: settled Tdie vs Ppkg

For each power step, the settled point is the ambient-normalized Tdie (`median(Tdie) - mean(Tair_in) + 20`) over the last 60 samples of that plateau, once it has held steady. The final plateau of each run is excluded: the acquisition harness ramps power until sustained thermal throttling aborts the whole run, so the last logged step was never confirmed stable. Both repeat runs are plotted as separate lines rather than merged, since their run-to-run gap isn't fully explained by the ambient difference between them.

In [41]:
plot_settled(baseline_runs)

### Takeaway

Both ASUS ROG Ryuo 120 runs throttle in the 150–170W range — short of the 7900X's rated 170W TDP (and well short of the 230W PPT ceiling the harness ramps toward). The Ryuo 120 is obviously incapable of providing decent cooling to a 170W-TDP CPU. The stock be quiet! Pure Loop 120 below demonstrates more or less the same performance — to no surprise, since both are similarly-sized 120mm AIOs with comparable radiator/pump/fan specs.

## Stock be quiet! Pure Loop 120 + stock CPU

In [42]:
stock_bequiet_runs = valid_runs[
    (~valid_runs['delidded']) & (valid_runs['loop_under_test'] == 'be quiet! Pure Loop 120')
]
plot_runs(stock_bequiet_runs)

### Stock be quiet! Pure Loop 120: settled Tdie vs Ppkg

Same method as the baseline: ambient-normalized settled Tdie per power step, final (throttle-abort) plateau of each run excluded, repeat runs plotted as separate lines.

In [43]:
plot_settled(stock_bequiet_runs)